# UC-09 · Generate the FAT report (docx, tex, pdf; en / pt)

**Use case:** [UC-09](../docs/product/use-cases.md) - *test complete, deliverable due.*

The library renders one `report_data` dictionary to three formats:

- `.docx` with `pump.utilities.report.ReportGenerator`;
- `.tex` (and its figures folder) with `LatexReportGenerator.generate_tex`;
- `.pdf` with `LatexReportGenerator.generate_pdf`, **only if a TeX engine is found**
  (`find_latex_engine()`; otherwise the PDF step is skipped and reported).

`language="en"` or `"pt"` selects the gettext catalogue and the template.  The
dictionary is built here by hand from library objects (design point, checker summary,
fitted curve, mechanical run, NPSH, comparison); the workbench does the same through
`pumpflow.binding.assemble_report_data`.  Everything is written to a temporary
directory.

**Acceptance:** the files exist and are non-empty, the docx re-opens and carries the
verdict in the right language, the LaTeX source carries the figures, and the PDF (when
a TeX engine exists) is a valid PDF.

## 1 · Setup and the test data

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import Q_, Fluid, Water, TestPoint, PerformanceCurve

# Illustrative water test of pump P-1001A (1480 rpm):
#          Q [m3/h]  Head [m]  P [kW]  eta [%]
P1001A = [
    (160.0, 125.37, 77.97, 70.03),
    (240.0, 119.11, 95.17, 81.82),
    (320.0, 110.79, 114.44, 84.44),
    (400.0, 100.01, 147.26, 74.00),
    (480.0, 86.72, 173.98, 65.13),
    (540.0, 75.60, 194.76, 57.14),
]
water = Fluid("Test water", density=Q_(1000.0, "kg/m**3"))


def build_curve(rows, fluid, speed, degree=3, head_k=1.0, power_k=1.0, eff_k=1.0):
    "Measured rows -> PerformanceCurve with Head / eta pinned on the points."
    points = [
        TestPoint(
            fluid=fluid,
            capacity=Q_(q, "m**3/h"),
            speed_of_rotation=speed,
            _head=Q_(h * head_k, "m"),
            _efficiency=Q_(e * eff_k, "percent"),
            breaking_power=Q_(p * power_k, "kW"),
        )
        for q, h, p, e in rows
    ]
    return PerformanceCurve(fluid, points, polynomial_degree=degree)

import math
import tempfile
from pathlib import Path

from docx import Document

from pump import (
    AcceptanceCriteria,
    Channel,
    DesignPoint,
    MechanicalRunChecker,
    MechanicalRunLog,
    NPSHrCurve,
    PerformanceChecker,
    Pressure,
    RunReading,
    check_npsh,
    npsh_available,
    plot_mechanical_run,
    plot_npsh,
)
from pump.compare import compare_curves, plot_comparison
from pump.utilities.report import ReportGenerator
from pump.utilities.report_latex import LatexReportGenerator, find_latex_engine

speed = Q_(1480, "rpm")

## 2 · The `report_data` builder

A function, because the chart streams (`BytesIO`) are consumed once by each writer.

In [ ]:
def make_mechanical():
    channels = (
        Channel("Filtered vibration", "vibration_filtered"),
        Channel("Unfiltered vibration", "vibration_unfiltered"),
        Channel("Bearing temperature", "bearing_temperature"),
    )
    readings = []
    for i in range(17):  # every 15 min for 4 h
        minutes = 15 * i
        readings.append(
            RunReading(
                f"{10 + minutes // 60:02d}:{minutes % 60:02d}",
                {
                    "Filtered vibration": Q_(0.05, "in/s"),
                    "Unfiltered vibration": Q_(0.09, "in/s"),
                    "Bearing temperature": Q_(150 + 0.5 * i, "degF"),
                },
            )
        )
    log = MechanicalRunLog(channels, readings)
    criteria = AcceptanceCriteria.load("api610:12").mechanical_run
    return {
        "result": MechanicalRunChecker(log, criteria).check(),
        "plot": plot_mechanical_run(log, criteria),
    }


def make_npsh(design):
    fluid = Water(Q_(20, "degC"))
    npsha = npsh_available(
        Pressure(Q_(0.2, "bar"), "g"), fluid, design.capacity,
        suction_id=Q_(150, "mm"), z_suction=Q_(1.0, "m"),
    )
    q = tuple(Q_(v, "m**3/h") for v in (160, 240, 320, 400))
    r = tuple(Q_(v, "m") for v in (2.0, 2.4, 3.0, 3.9))
    curve = NPSHrCurve(q, r, degree=2)
    result = check_npsh(npsha, curve.at_rated(design.capacity))
    return {"result": result, "plot": plot_npsh(curve, npsha, design.capacity),
            "table": list(zip(curve.q, curve.npshr))}


def make_report_data():
    curve = build_curve(P1001A, water, speed)
    design = DesignPoint(
        fluid=water,
        capacity=Q_(320, "m**3/h"),
        differential_head=Q_(110, "m"),
        breaking_power=Q_(115, "kW"),
    )
    criteria = AcceptanceCriteria.load("api610:12")
    checker = PerformanceChecker(design, curve, criteria=criteria)

    pred_h = curve.predict_head(design.capacity)
    pred_p = curve.predict_breaking_power(design.capacity)
    passed = bool(
        checker.minimum_head <= pred_h <= checker.maximum_head
        and pred_p <= checker.maximum_breaking_power
    )

    curve_b = build_curve(P1001A, water, speed, head_k=0.98, power_k=1.03, eff_k=0.98 / 1.03)
    curves = {"P-1001A": curve, "P-1001B": curve_b}
    comparison = {
        "table": compare_curves(curves, [design.capacity]),
        "plot": plot_comparison(curves, rated_q=design.capacity),
        "at": [design.capacity],
    }
    entry = {
        "test_summary": checker.report_summary,
        "test_data": curve.test_data,
        "verdict": passed,
        "Performance Curve": curve.plot_performance_curve(
            capacity=design.capacity, return_io=True
        ),
        "Mechanical Run": make_mechanical(),
        "NPSH": make_npsh(design),
    }
    return {
        "equipment_description": {
            "TAG": "P-1001A",
            "Manufacturer": "Demo Pumps",
            "Service": "Cooling water",
        },
        "design_point": design,
        "test_data": {"P-1001A": entry},
        "acceptance": {
            "label": criteria.label,
            "verified": bool(criteria.verified),
            "unverified": list(criteria.unverified),
        },
        "comparison": comparison,
    }


probe = make_report_data()
print("sections:", sorted(probe["test_data"]["P-1001A"]), "+ comparison")
print("verdict :", probe["test_data"]["P-1001A"]["verdict"])

## 3 · Write every format

Each language gets a `.docx` and a `.tex`; the PDF is compiled only when a TeX engine
is available, and a compilation problem in the local TeX installation is reported
rather than raised (the `.tex` assertions still cover the content).

In [ ]:
engine = find_latex_engine()
print("TeX engine:", engine)

docx_text, tex_text, sizes = {}, {}, {}
pdf_status = "skipped (no TeX engine found)" if engine is None else "not run"
pdf_bytes = None
n_figures = {}

with tempfile.TemporaryDirectory() as tmp_name:
    tmp = Path(tmp_name)
    for lang in ("en", "pt"):
        docx_path = Path(
            ReportGenerator(language=lang).generate_report(
                make_report_data(), output_file=str(tmp / f"report_{lang}.docx")
            )
        )
        tex_path = Path(
            LatexReportGenerator(language=lang).generate_tex(
                make_report_data(), str(tmp / f"report_{lang}.tex")
            )
        )
        doc = Document(str(docx_path))
        docx_text[lang] = "\n".join(p.text for p in doc.paragraphs)
        tex_text[lang] = tex_path.read_text(encoding="utf-8")
        figs = tmp / f"report_{lang}_figs"
        n_figures[lang] = len(list(figs.glob("*.png"))) if figs.is_dir() else 0
        sizes[lang] = (docx_path.stat().st_size, tex_path.stat().st_size)

    if engine is not None:
        try:
            pdf_path = LatexReportGenerator(language="en").generate_pdf(
                make_report_data(), str(tmp / "report_en.pdf")
            )
            pdf_bytes = Path(pdf_path).read_bytes()
            pdf_status = f"compiled ({len(pdf_bytes)} bytes)"
        except RuntimeError as exc:
            pdf_status = "TeX engine found but the build failed: " + str(exc)[:150]

print("sizes (docx, tex):", sizes)
print("figures per language:", n_figures)
print("PDF:", pdf_status)

## 4 · Acceptance

In [ ]:
for lang in ("en", "pt"):
    assert sizes[lang][0] > 10_000, "docx is empty"
    assert sizes[lang][1] > 2_000, "tex is empty"
    assert n_figures[lang] >= 3, "performance, NPSH/mechanical and comparison figures"

# the docx carries the pump tag and the localised verdict (API 610 12th ed.
# Table 16 is checked against the text, so the verdict is not provisional)
assert "P-1001A" in docx_text["en"] and "P-1001A" in docx_text["pt"]
assert "Overall verdict" in docx_text["en"] and "APPROVED" in docx_text["en"]
assert "APPROVED (unverified criteria)" not in docx_text["en"]
assert "Resultado geral" in docx_text["pt"] and "APROVADO" in docx_text["pt"]
assert "Dados do Teste" in docx_text["pt"]

# the LaTeX source is localised too, and lists the comparison section
assert r"\section{Introduction}" in tex_text["en"]
assert "Introdu" in tex_text["pt"] and "brazilian" in tex_text["pt"]
assert "Comparison" in tex_text["en"]
assert r"\includegraphics" in tex_text["en"]

# the PDF exists only when a TeX engine is installed; if built it is a real PDF
if pdf_bytes is not None:
    assert pdf_bytes.startswith(b"%PDF")
print("UC-09 acceptance OK  | PDF:", pdf_status)